# Bronze: Claimant Count
**Source:** MHCLG LA Revenue Account Outturn
**Purpose:** Load raw CSV into Bronze Delta table
**Date loaded:** 7th April 2026
**Rows expected:** approximately 300 (one per local authority)

In [9]:
%pip install xlrd

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 45, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [11]:
import os
files = os.listdir('/lakehouse/default/Files/')
for f in files:
    print(f)

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 49, Finished, Available, Finished, False)

Index of Multiple Deprivation (IMD).csv
REFERENCE
claimant_countJan.xls
ks4_attainment.csv
la_revenue_expenditure.csv


In [12]:
pdf = pd.read_excel(
    '/lakehouse/default/Files/claimant_countJan.xls',
    sheet_name='CC01',
    skiprows=4
)

print(f'Rows: {len(pdf)}')
print(f'Columns: {list(pdf.columns)}')
pdf.head(5)

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 51, Finished, Available, Finished, False)

Rows: 405
Columns: ['Geography', 'Geography code', 'Number of men1', 'Number of women1', 'Number of people1', 'Proportion of men2', 'Proportion of women2', 'Proportion of people2', 'Men:\nchange on year', 'Women:\nchange on year', 'People:\nchange on year', 'Men:\nproportion change on year2', 'Women:\nproportion change on year2', 'People:\nproportion change on year2']


,Geography,Geography code,Number of men1,Number of women1,Number of people1,Proportion of men2,Proportion of women2,Proportion of people2,Men:\nchange on year,Women:\nchange on year,People:\nchange on year,Men:\nproportion change on year2,Women:\nproportion change on year2,People:\nproportion change on year2
0,United Kingdom,K02000001,934665,732320,1666985,4.4,3.3,3.8,-6050,-24330,-30380,0.0,-0.1,-0.1
1,Great Britain,K03000001,915510,716295,1631805,4.4,3.3,3.9,-4530,-23225,-27755,0.0,-0.1,-0.1
2,England,E92000001,816255,647120,1463375,4.5,3.5,4.0,-3465,-20610,-24075,0.0,-0.1,-0.1
3,North East,E12000001,37240,25240,62480,4.4,2.9,3.7,-1575,-2185,-3760,-0.2,-0.3,-0.2
4,County Durham,E06000047,6095,4260,10355,3.8,2.5,3.1,-155,-260,-415,-0.1,-0.2,-0.1


In [16]:
print(pdf[['Geography', 'Geography code', 'Proportion of people2']].head(10))

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 56, Finished, Available, Finished, False)

              Geography Geography code  Proportion of people2
0        United Kingdom      K02000001                    3.8
1         Great Britain      K03000001                    3.9
2               England      E92000001                    4.0
3            North East      E12000001                    3.7
4         County Durham      E06000047                    3.1
5            Darlington      E06000005                    3.5
6            Hartlepool      E06000001                    4.4
7         Middlesbrough      E06000002                    4.9
8        Northumberland      E06000057                    2.9
9  Redcar and Cleveland      E06000003                    3.7


In [18]:
pdf = pdf.rename(columns={
    'Geography': 'geography',
    'Geography code': 'geography_code',
    'Number of men1': 'claimant_count_men',
    'Number of women1': 'claimant_count_women',
    'Number of people1': 'claimant_count_total',
    'Proportion of men2': 'claimant_rate_men',
    'Proportion of women2': 'claimant_rate_women',
    'Proportion of people2': 'claimant_rate_total',
    'Men:\nchange on year': 'change_on_year_men',
    'Women:\nchange on year': 'change_on_year_women',
    'People:\nchange on year': 'change_on_year_total',
    'Men:\nproportion change on year2': 'proportion_change_men',
    'Women:\nproportion change on year2': 'proportion_change_women',
    'People:\nproportion change on year2': 'proportion_change_total'
})

print('Columns renamed:')
print(list(pdf.columns))

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 58, Finished, Available, Finished, False)

Columns renamed:
['geography', 'geography_code', 'claimant_count_men', 'claimant_count_women', 'claimant_count_total', 'claimant_rate_men', 'claimant_rate_women', 'claimant_rate_total', 'change_on_year_men', 'change_on_year_women', 'change_on_year_total', 'proportion_change_men', 'proportion_change_women', 'proportion_change_total']


In [19]:
df = spark.createDataFrame(pdf)
df.write.format('delta').mode('overwrite').saveAsTable('bronze_claimant_count')
print('Bronze claimant count table created successfully')

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 59, Finished, Available, Finished, False)

Bronze claimant count table created successfully


In [20]:
spark.sql("SELECT * FROM bronze_claimant_count LIMIT 5").show(truncate=False)

StatementMeta(, bb65c34b-3a6d-41f9-832d-4eb2c8af544a, 60, Finished, Available, Finished, False)

+--------------+--------------+------------------+--------------------+--------------------+-----------------+-------------------+-------------------+------------------+--------------------+--------------------+---------------------+-----------------------+-----------------------+
|geography     |geography_code|claimant_count_men|claimant_count_women|claimant_count_total|claimant_rate_men|claimant_rate_women|claimant_rate_total|change_on_year_men|change_on_year_women|change_on_year_total|proportion_change_men|proportion_change_women|proportion_change_total|
+--------------+--------------+------------------+--------------------+--------------------+-----------------+-------------------+-------------------+------------------+--------------------+--------------------+---------------------+-----------------------+-----------------------+
|Merthyr Tydfil|W06000024     |690               |505                 |1195                |3.9              |2.7                |3.3                |35  